In [28]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [29]:
import pandas as pd
import pandas as pd
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS



In [30]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [31]:
# Question 1

freq = df['answer'].value_counts().sort_index()
print("Frequency Distribution:")
print(freq)

most_frequent = freq.max()
least_frequent = freq.min()
total = most_frequent + least_frequent

print(f"\nMost frequent: {freq.idxmax()} = {most_frequent}")
print(f"Least frequent: {freq.idxmin()} = {least_frequent}")
print(f"Q1 - Sum = {total}")

Frequency Distribution:
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Most frequent: B = 490
Least frequent: E = 324
Q1 - Sum = 814


In [32]:
# Cleaing lowercase and removing punctuation
df['cleaned'] = df['prompt'].str.lower().str.translate(str.maketrans('', '', string.punctuation))


In [33]:
# Question 2
all_words = df['cleaned'].str.split().explode()
vocab_size = all_words.nunique()
print(f"Q2 - Total unique words (vocabulary size): {vocab_size}")

Q2 - Total unique words (vocabulary size): 859


In [34]:
# Question 3
row1 = df[df['id'] == 1]['cleaned'].values[0]
words_row1 = row1.split()
filtered = [w for w in words_row1 if w not in ENGLISH_STOP_WORDS]
print(f"\nQ3 - Row ID 1 words after stop word removal: {len(filtered)}")
print(f"Remaining words: {filtered}")


Q3 - Row ID 1 words after stop word removal: 13
Remaining words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


In [35]:
# Question 4
all_texts = []
for _, row in df.iterrows():
    combined = ' '.join([str(row['prompt']), str(row['A']), str(row['B']),
                         str(row['C']), str(row['D']), str(row['E'])])
    all_texts.append(combined)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(all_texts)
print(f"Q4 - TF-IDF vocabulary size: {len(vectorizer.vocabulary_)}")

Q4 - TF-IDF vocabulary size: 2762


In [36]:
# Question 5

row1 = df[df['id'] == 1].iloc[0]
prompt_vec = vectorizer.transform([str(row1['prompt'])])
optionA_vec = vectorizer.transform([str(row1['A'])])
sim = cosine_similarity(prompt_vec, optionA_vec)[0][0]
print(f"\nQ5 - Cosine similarity (prompt vs A) for Row ID 1: {round(sim, 4)}")


Q5 - Cosine similarity (prompt vs A) for Row ID 1: 0.272


In [37]:
#  Question 6

correct = 0
for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for opt in ['A', 'B', 'C', 'D', 'E']:
        opt_vec = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    predicted = max(sims, key=sims.get)
    if predicted == row['answer']:
        correct += 1

pct = (correct / len(df)) * 100
print(f"\nQ6 - Accuracy (highest cosine sim = correct answer): {round(pct, 2)}%")


Q6 - Accuracy (highest cosine sim = correct answer): 13.55%


In [43]:
#  Question 7

print(f"Q7 - MAP@3 (truth=C, pred=C A B): {map_at_3('C', ['C','A','B'])}")


Q7 - MAP@3 (truth=C, pred=C A B): 1.0


In [44]:
#  Question 8

def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return round(1.0 / (k + 1), 4)
    return 0.0

print(f"\nQ8 - MAP@3 (truth=B, pred=D B E): {map_at_3('B', ['D','B','E'])}")


Q8 - MAP@3 (truth=B, pred=D B E): 0.5


In [40]:
def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0


In [41]:
#  Question 9

freq = df['answer'].value_counts()
top3 = freq.index[:3].tolist()   # e.g. ['B', 'A', 'C']
print(f"Top 3 most frequent answers: {top3}")

scores = [map_at_3(row['answer'], top3) for _, row in df.iterrows()]
print(f"Q9 - Majority Class Baseline MAP@3: {round(sum(scores)/len(scores), 4)}")


Top 3 most frequent answers: ['B', 'C', 'A']
Q9 - Majority Class Baseline MAP@3: 0.4213


In [42]:
#  Question 10

all_texts = []
for _, row in df.iterrows():
    combined = ' '.join([str(row[c]) for c in ['prompt','A','B','C','D','E']])
    all_texts.append(combined)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(all_texts)

scores_tfidf = []
for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for opt in ['A','B','C','D','E']:
        opt_vec = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    top3_pred = sorted(sims, key=sims.get, reverse=True)[:3]
    scores_tfidf.append(map_at_3(row['answer'], top3_pred))

print(f"Q10 - TF-IDF Pipeline MAP@3: {round(sum(scores_tfidf)/len(scores_tfidf), 4)}")

Q10 - TF-IDF Pipeline MAP@3: 0.2962
